# 03 — Unreddened Template

Load the intrinsic QSO spectral template, redshift it to each QSO, compute synthetic photometry
via `synphot`, and overlay it on the observed SED (median-flux-scaled) for visual comparison.

Consolidates `scripts/seds/unred.py`, `DESI_SEDs_unred.py`, `PSAWG_SEDs_unred.py`,
`PSG_SEDs_unred.py`, `W2M_legacy_SEDs_unred.py` into one parameterized function.

**Important — reconciling two code paths:** the `quasar_unred` PyPI package
(`load_template`, `extinguish`, `fit_composite`, `find_ebv`, `mc_spec`) is imported in
`DESI_SEDs_unred.py`, `PSAWG_SEDs_unred.py`, `PSG_SEDs_unred.py`, but **none of those scripts
actually call any of its functions** — the import is vestigial. All "unred" scripts (including
this notebook, faithfully porting their behavior) only do template overlay + median-flux scaling
for visual comparison; **no script anywhere in this repo actually fits E(B-V) via
`quasar_unred.find_ebv`**. The patchy-obscuration model in the project's `CLAUDE.md` is a stated
goal, not yet an implemented step. `find_ebv`/`mc_spec` are exposed below as a documented next
step but are not yet wired into the pipeline — flag this to the user before treating any current
E(B-V) values as model-fit results rather than the DESI catalog's own EBV column.

**Also note:** `scripts/seds/COMBINED_SEDs_unred.py` references `data/matched/COMBINED_matched.csv`
(does not exist) and `Jmag_2mass`/`Hmag_2mass`/`Kmag_2mass` columns (not present in any current
matched CSV — the DESI schema uses `Jmag`/`Hmag`/`Kmag` from AllWISE's paired 2MASS-like columns).
This script appears broken/aspirational and is not ported here; use `DESI_COMBINED_matched.csv`
via the DESI-schema path if a "combined" unred pass is needed.

**Inputs:** `templates/qso_template.txt`, `data/filters/*.dat`, `data/matched/*_matched.csv`
**Script references:** `scripts/seds/unred.py`, `*_SEDs_unred.py`

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import yaml
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import ascii
import astropy.units as u

import synphot
from synphot import SourceSpectrum, SpectralElement
from synphot import units as su
from synphot.models import Empirical1D
from synphot.observation import Observation

from quasar_unred import load_template, extinguish, fit_composite, find_ebv, mc_spec

BASE_DIR = Path.cwd().parent
with open(BASE_DIR / "config" / "qso_params.yaml") as f:
    PARAMS = yaml.safe_load(f)

DATA_MATCHED = BASE_DIR / "data" / "matched"
FILTER_DIR = BASE_DIR / "data" / "filters"
TEMPLATE_FILE = BASE_DIR / "templates" / "qso_template.txt"

WAVELENGTHS = PARAMS["band_wavelengths_AA"]
VEGA_TO_AB = PARAMS["vega_to_ab_offsets"]
FLAM_MAX_GALEX_PS1 = PARAMS["flux_outlier_thresholds_flam"]["galex_panstarrs"] * su.FLAM
FLAM_MAX_UKIDSS = PARAMS["flux_outlier_thresholds_flam"]["ukidss"] * su.FLAM


def mag_arr(col):
    if hasattr(col, 'filled'):
        return np.array(col.filled(np.nan), dtype=float)
    arr = np.array(col, dtype=str)
    arr[arr == '--'] = 'nan'
    return arr.astype(float)


# load_template() from quasar_unred ships its own default template; we use the project's
# templates/qso_template.txt instead, so load it directly with astropy.io.ascii for full control
# over path handling (matches templateWave/templateFlux structure load_template() would return).
_spec = ascii.read(str(TEMPLATE_FILE))
templateWave = _spec['col1']
templateFlux = _spec['col2']

## DESI-schema template overlay (DESI, PSAWG, PSG)

Source: `scripts/seds/DESI_SEDs_unred.py`, `PSAWG_SEDs_unred.py`, `PSG_SEDs_unred.py`.
These three differ only in input CSV (and PSAWG/PSG lack UKIDSS bands, per their matched-CSV
schema) so they share one function. Template synthetic photometry uses only the bands with
filter curves available (GALEX + PanSTARRS + UKIDSS; no WISE filter curve is used here, matching
the original scripts).

In [ ]:
DESI_UNRED_FILT_FILES = [
    "GALEX_GALEX.FUV.dat", "GALEX_GALEX.NUV.dat",
    "PAN-STARRS_PS1.g.dat", "PAN-STARRS_PS1.r.dat", "PAN-STARRS_PS1.i.dat",
    "PAN-STARRS_PS1.z.dat", "PAN-STARRS_PS1.y.dat",
    "UKIRT_UKIDSS.Y.dat", "UKIRT_UKIDSS.J.dat", "UKIRT_UKIDSS.H.dat", "UKIRT_UKIDSS.K.dat",
]
DESI_UNRED_LAM_TEMPLATE = np.array([WAVELENGTHS[b] for b in
    ['FUV', 'NUV', 'g', 'r', 'i', 'z', 'y', 'Y', 'J', 'H', 'K']]) * u.AA


def desi_unred_flam(table, has_wise=True):
    """FLAM for the DESI SED schema. UKIDSS is always computed (needed for template-scale
    fitting, matching all three original scripts); has_wise only gates whether WISE bands
    (not used for template scaling in any of the original scripts) are computed/plotted."""
    flam = {}
    flam['FUV'] = (mag_arr(table['FUVmag']) * u.ABmag).to(su.FLAM, u.spectral_density(WAVELENGTHS['FUV'] * u.AA))
    flam['FUV'][flam['FUV'] > FLAM_MAX_GALEX_PS1] = np.nan
    flam['NUV'] = (mag_arr(table['NUVmag']) * u.ABmag).to(su.FLAM, u.spectral_density(WAVELENGTHS['NUV'] * u.AA))
    flam['NUV'][flam['NUV'] > FLAM_MAX_GALEX_PS1] = np.nan
    for band, col in [('g', 'gmag'), ('r', 'rmag'), ('i', 'imag'), ('z', 'zmag'), ('y', 'ymag')]:
        f = (mag_arr(table[col]) * u.ABmag).to(su.FLAM, u.spectral_density(WAVELENGTHS[band] * u.AA))
        f[f > FLAM_MAX_GALEX_PS1] = np.nan
        flam[band] = f
    for band, col in [('Y', 'yAperMag3'), ('J', 'j_1AperMag3'), ('H', 'hAperMag3'), ('K', 'kAperMag3')]:
        f = ((mag_arr(table[col]) + VEGA_TO_AB[band]) * u.ABmag).to(su.FLAM, u.spectral_density(WAVELENGTHS[band] * u.AA))
        f[f > FLAM_MAX_UKIDSS] = np.nan
        flam[band] = f
    if has_wise:
        for band, col in [('W1', 'W1mag'), ('W2', 'W2mag'), ('W3', 'W3mag'), ('W4', 'W4mag')]:
            flam[band] = ((mag_arr(table[col]) + VEGA_TO_AB[band]) * u.ABmag).to(su.FLAM, u.spectral_density(WAVELENGTHS[band] * u.AA))
    return flam


def plot_desi_unred(csv_path, has_wise=True, only_uv_excess=True, ebv_min=None):
    if ebv_min is None:
        ebv_min = PARAMS["uv_excess"]["ebv_min"]
    table = Table.read(csv_path, format='csv')
    flam = desi_unred_flam(table, has_wise=has_wise)
    targetID = table['TARGETID']
    redshift = table['Z']
    ebv = mag_arr(table['EBV'])

    plot_bands = ['FUV', 'NUV', 'g', 'r', 'i', 'z', 'y', 'Y', 'J', 'H', 'K']
    if has_wise:
        plot_bands += ['W1', 'W2', 'W3', 'W4']
    lam_plot = np.array([WAVELENGTHS[b] for b in plot_bands]) * u.AA

    f_fn = flam['FUV'] - flam['NUV']
    f_ng = flam['NUV'] - flam['g']
    f_gr = flam['g'] - flam['r']

    def is_uv_excess(index):
        upturn = ((f_ng[index] > 0 and f_gr[index] < 0) or (f_fn[index] > 0 and f_ng[index] < 0))
        return upturn and (flam['FUV'][index] > flam['NUV'][index]) and (ebv[index] > ebv_min)

    for index, name in enumerate(targetID):
        if only_uv_excess and not is_uv_excess(index):
            continue
        zsp = redshift[index]

        plt.figure()
        values = [flam[b].value[index] for b in plot_bands]
        plt.plot(lam_plot.value / (1 + zsp), values, marker='o', label=name)

        sp = SourceSpectrum(Empirical1D, points=templateWave * u.AA,
                             lookup_table=templateFlux * su.FLAM, z=zsp)
        synth_flx = []
        for filt_file in DESI_UNRED_FILT_FILES:
            bp = SpectralElement.from_file(str(FILTER_DIR / filt_file))
            try:
                obs = Observation(sp, bp, force='extrap')
                synth_flx.append(obs.effstim('flam').value)
            except (synphot.exceptions.DisjointError, synphot.exceptions.SynphotError):
                synth_flx.append(np.nan)
        synth_flx = np.array(synth_flx)

        obs_flam = np.array([flam[b].value[index] for b in
                              ['FUV', 'NUV', 'g', 'r', 'i', 'z', 'y', 'Y', 'J', 'H', 'K']])
        valid = np.isfinite(obs_flam) & np.isfinite(synth_flx) & (synth_flx > 0)
        scale = np.nanmedian(obs_flam[valid] / synth_flx[valid]) if valid.any() else 1.0

        plt.plot(templateWave, scale * templateFlux, color='gray', alpha=0.6, label='QSO template')
        plt.scatter(DESI_UNRED_LAM_TEMPLATE.value / (1 + zsp), scale * synth_flx,
                    color='orange', marker='s', zorder=5, label='template synth phot')

        plt.xscale('log')
        plt.yscale('log')
        plt.xlim(700, 7000)
        plt.ylim(1e-18, 1e-15)
        plt.title(f"E(B-V) = {ebv[index]:.3f}")
        plt.legend()
        plt.show()


# DESI
plot_desi_unred(DATA_MATCHED / "UKPSAWG_matched.csv", has_wise=True)

# PSAWG, PSG: same schema minus WISE columns
plot_desi_unred(DATA_MATCHED / "PSAWG_matched.csv", has_wise=False)
plot_desi_unred(DATA_MATCHED / "PSG_matched.csv", has_wise=False)

## W2M-legacy template overlay

Source: `scripts/seds/W2M_legacy_SEDs_unred.py`. Different schema (`PS1_gmag`...`PS1_ymag`,
`j_m_2mass`/`h_m_2mass`/`k_m_2mass`, `w1mpro`...`w4mpro`) from `data/matched/W2M_legacy_matched.csv`
— a distinct file from `W2M_legacy_COMBINED_matched.csv` used elsewhere. Template synthetic
photometry here uses GALEX + PanSTARRS + WISE filters (no UKIDSS, since this schema has none).

In [ ]:
W2M_LEGACY_FILT_FILES = [
    "GALEX_GALEX.FUV.dat", "GALEX_GALEX.NUV.dat",
    "PAN-STARRS_PS1.g.dat", "PAN-STARRS_PS1.r.dat", "PAN-STARRS_PS1.i.dat",
    "PAN-STARRS_PS1.z.dat", "PAN-STARRS_PS1.y.dat",
    "WISE_WISE.W1.dat", "WISE_WISE.W2.dat", "WISE_WISE.W3.dat", "WISE_WISE.W4.dat",
]
W2M_LEGACY_LAM_ALL = np.array([1549, 2303, 4810, 6170, 7520, 8660, 9620,
                                12350, 16620, 21590, 33680, 46180, 120820, 221940])
W2M_LEGACY_LAM_TEMPLATE = np.array([1549, 2303, 4810, 6170, 7520, 8660, 9620,
                                     33680, 46180, 120820, 221940]) * u.AA

w2m_legacy_table = Table.read(DATA_MATCHED / "W2M_legacy_matched.csv", format='csv')

flam_fuv = (mag_arr(w2m_legacy_table['FUVmag']) * u.ABmag).to(su.FLAM, u.spectral_density(1549 * u.AA))
flam_fuv[flam_fuv > FLAM_MAX_GALEX_PS1] = np.nan
flam_nuv = (mag_arr(w2m_legacy_table['NUVmag']) * u.ABmag).to(su.FLAM, u.spectral_density(2303 * u.AA))
flam_nuv[flam_nuv > FLAM_MAX_GALEX_PS1] = np.nan
flam_g = (mag_arr(w2m_legacy_table['PS1_gmag']) * u.ABmag).to(su.FLAM, u.spectral_density(4810 * u.AA))
flam_g[flam_g > FLAM_MAX_GALEX_PS1] = np.nan
flam_r = (mag_arr(w2m_legacy_table['PS1_rmag']) * u.ABmag).to(su.FLAM, u.spectral_density(6170 * u.AA))
flam_r[flam_r > FLAM_MAX_GALEX_PS1] = np.nan
flam_i = (mag_arr(w2m_legacy_table['PS1_imag']) * u.ABmag).to(su.FLAM, u.spectral_density(7520 * u.AA))
flam_i[flam_i > FLAM_MAX_GALEX_PS1] = np.nan
flam_z = (mag_arr(w2m_legacy_table['PS1_zmag']) * u.ABmag).to(su.FLAM, u.spectral_density(8660 * u.AA))
flam_z[flam_z > FLAM_MAX_GALEX_PS1] = np.nan
flam_y = (mag_arr(w2m_legacy_table['PS1_ymag']) * u.ABmag).to(su.FLAM, u.spectral_density(9620 * u.AA))
flam_y[flam_y > FLAM_MAX_GALEX_PS1] = np.nan

flam_J2m = ((mag_arr(w2m_legacy_table['j_m_2mass']) + 0.894) * u.ABmag).to(su.FLAM, u.spectral_density(12350 * u.AA))
flam_J2m[flam_J2m > FLAM_MAX_UKIDSS] = np.nan
flam_H2m = ((mag_arr(w2m_legacy_table['h_m_2mass']) + 1.374) * u.ABmag).to(su.FLAM, u.spectral_density(16620 * u.AA))
flam_H2m[flam_H2m > FLAM_MAX_UKIDSS] = np.nan
flam_K2m = ((mag_arr(w2m_legacy_table['k_m_2mass']) + 1.839) * u.ABmag).to(su.FLAM, u.spectral_density(21590 * u.AA))
flam_K2m[flam_K2m > FLAM_MAX_UKIDSS] = np.nan

flam_w1 = ((mag_arr(w2m_legacy_table['w1mpro']) + VEGA_TO_AB['W1']) * u.ABmag).to(su.FLAM, u.spectral_density(33680 * u.AA))
flam_w2 = ((mag_arr(w2m_legacy_table['w2mpro']) + VEGA_TO_AB['W2']) * u.ABmag).to(su.FLAM, u.spectral_density(46180 * u.AA))
flam_w3 = ((mag_arr(w2m_legacy_table['w3mpro']) + VEGA_TO_AB['W3']) * u.ABmag).to(su.FLAM, u.spectral_density(120820 * u.AA))
flam_w4 = ((mag_arr(w2m_legacy_table['w4mpro']) + VEGA_TO_AB['W4']) * u.ABmag).to(su.FLAM, u.spectral_density(221940 * u.AA))

w2m_legacy_names = np.array(w2m_legacy_table['designation'])
w2m_legacy_z = mag_arr(w2m_legacy_table['zsp'])


def has_uv(index):
    return np.isfinite(flam_fuv.value[index]) or np.isfinite(flam_nuv.value[index])


for index, name in enumerate(w2m_legacy_names):
    if not has_uv(index):
        continue
    zsp = w2m_legacy_z[index]

    flam_all = np.array([
        flam_fuv.value[index], flam_nuv.value[index],
        flam_g.value[index], flam_r.value[index],
        flam_i.value[index], flam_z.value[index], flam_y.value[index],
        flam_J2m.value[index], flam_H2m.value[index], flam_K2m.value[index],
        flam_w1.value[index], flam_w2.value[index], flam_w3.value[index], flam_w4.value[index],
    ])

    plt.figure()
    plt.plot(W2M_LEGACY_LAM_ALL / (1 + zsp), flam_all, marker='o', linestyle='-', label=str(name))

    sp = SourceSpectrum(Empirical1D, points=templateWave * u.AA,
                         lookup_table=templateFlux * su.FLAM, z=zsp)
    synth_flx = []
    for filt_file in W2M_LEGACY_FILT_FILES:
        bp = SpectralElement.from_file(str(FILTER_DIR / filt_file))
        try:
            obs = Observation(sp, bp, force='extrap')
            synth_flx.append(obs.effstim('flam').value)
        except (synphot.exceptions.DisjointError, synphot.exceptions.SynphotError):
            synth_flx.append(np.nan)
    synth_flx = np.array(synth_flx)

    obs_for_scale = np.array([
        flam_fuv.value[index], flam_nuv.value[index],
        flam_g.value[index], flam_r.value[index], flam_i.value[index],
        flam_z.value[index], flam_y.value[index],
        flam_w1.value[index], flam_w2.value[index], flam_w3.value[index], flam_w4.value[index],
    ])
    valid = np.isfinite(obs_for_scale) & np.isfinite(synth_flx) & (synth_flx > 0)
    scale = np.nanmedian(obs_for_scale[valid] / synth_flx[valid]) if valid.any() else 1.0

    plt.plot(templateWave, scale * templateFlux, color='gray', alpha=0.6, label='QSO template')
    lam_tpl_rest = W2M_LEGACY_LAM_TEMPLATE.value / (1 + zsp)
    valid_tpl = np.isfinite(synth_flx)
    plt.scatter(lam_tpl_rest[valid_tpl], scale * synth_flx[valid_tpl],
                color='orange', marker='s', zorder=5, label='template synth phot')

    plt.xscale('log')
    plt.yscale('log')
    plt.ylim(1e-18, 1e-15)
    plt.xlabel('Rest Wavelength (Å)')
    plt.ylabel(r'$F_\lambda$ (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)')
    plt.title(f'{name}   z = {zsp:.4f}')
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## Next step (not yet run): fitting E(B-V) with `quasar_unred`

None of the sections above actually fit E(B-V) — they only overlay a median-flux-scaled
template for visual comparison. The `quasar_unred` package's `fit_composite` + `find_ebv` (+
`mc_spec` for uncertainty) would let this notebook go further: fit a scaling ratio between the
template and the observed photometry, then solve for the E(B-V) that best reconciles the two
under the patchy-obscuration model in `CLAUDE.md`. Sketch (not executed — no script in this repo
has done this yet, so there's no verified reference behavior to port):

```python
# srat = fit_composite(templateWave, templateFlux, wphot_rest, obs_flam)
# ebv_fit = find_ebv(templateWave, templateFlux, wphot_rest, obs_flam, srat)
# ebv_fit_err = mc_spec(templateWave, templateFlux, wphot_rest, obs_flam, obs_err, srat)
```